# MAE 6246 - Week 2 Python Example

## Coordinates, modes, singular values, and Jordan blocks

This notebook is the computational companion to the Week 2 lecture. It emphasizes demonstrations that are more informative in Python than through lengthy hand calculations:

1. transform a coupled mechanical system into modal coordinates;
2. compare coupled physical motion with independent modal motion;
3. use the SVD to reveal rank, directional amplification, and conditioning;
4. compute the exact Jordan decomposition of a supplied square matrix;
5. construct higher-order Jordan blocks and verify their Jordan chains; and
6. compute the exponential of a Jordan block from its finite nilpotent series.

Jordan form is valuable for understanding structure, but it is not numerically stable to infer from a general floating-point matrix. Here we construct matrices whose Jordan structure is known exactly.

In [ ]:
from math import factorial

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import block_diag, expm

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

## 1. One operator in two coordinate systems

Consider the stiffness operator

$$
K=\begin{bmatrix}2&-1\\-1&2\end{bmatrix}.
$$

Because $K$ is real and symmetric, `numpy.linalg.eigh` returns real eigenvalues and an orthonormal eigenvector matrix $P$. Physical and modal coordinates satisfy

$$
q=P\eta,\qquad \eta=P^Tq,
$$

and the modal stiffness matrix is $K_m=P^TKP$.

In [ ]:
K = np.array([[2.0, -1.0], [-1.0, 2.0]])

eigenvalues, P = np.linalg.eigh(K)
K_modal = P.T @ K @ P

q = np.array([1.0, 0.25])
eta = P.T @ q
q_reconstructed = P @ eta

print("eigenvalues =", eigenvalues)
print("eigenvector matrix P =\n", P)
print("\nP^T P =\n", P.T @ P)
print("\nK in modal coordinates =\n", K_modal)
print("\nphysical q =", q)
print("modal eta =", eta)
print("reconstructed q =", q_reconstructed)

assert np.allclose(P.T @ P, np.eye(2))
assert np.allclose(K_modal, np.diag(eigenvalues))
assert np.allclose(q_reconstructed, q)

The coordinate change has not altered the physical operator. It has selected directions in which the operator acts independently. The signs of computed eigenvectors may differ from a hand calculation; multiplying an eigenvector by $-1$ does not change its eigenspace.

## 2. Coupled motion and modal motion

For two unit masses with no damping,

$$
\ddot q+Kq=0.
$$

In modal coordinates, the equations become

$$
\ddot\eta+K_m\eta=0.
$$

The physical coordinates generally contain both natural frequencies, while each modal coordinate contains only one.

In [ ]:
def coupled_oscillator_rhs(t, state):
    q = state[:2]
    q_dot = state[2:]
    return np.concatenate((q_dot, -K @ q))


t_eval = np.linspace(0.0, 20.0, 2001)
initial_state = np.array([1.0, 0.0, 0.0, 0.0])
solution = solve_ivp(
    coupled_oscillator_rhs,
    (t_eval[0], t_eval[-1]),
    initial_state,
    t_eval=t_eval,
    rtol=1e-10,
    atol=1e-12,
)

q_history = solution.y[:2]
eta_history = P.T @ q_history

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(t_eval, q_history[0], label=r"$q_1$")
axes[0].plot(t_eval, q_history[1], label=r"$q_2$")
axes[0].set_ylabel("physical displacement")
axes[0].set_title("Coupled physical coordinates")
axes[0].legend()

axes[1].plot(t_eval, eta_history[0], label=r"$\eta_1$")
axes[1].plot(t_eval, eta_history[1], label=r"$\eta_2$")
axes[1].set_xlabel("time")
axes[1].set_ylabel("modal displacement")
axes[1].set_title("Independent modal coordinates")
axes[1].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3. Rank and null space from the SVD

For $A=U\Sigma V^T$, the number of singular values above a selected tolerance is the numerical rank. Right singular vectors associated with zero singular values span the null space.

In [ ]:
A_rank_deficient = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
])

U, singular_values, Vt = np.linalg.svd(A_rank_deficient, full_matrices=True)
tolerance = (
    np.finfo(float).eps
    * max(A_rank_deficient.shape)
    * singular_values[0]
)
rank = np.sum(singular_values > tolerance)
null_basis = Vt[rank:].T

print("singular values =", singular_values)
print("numerical tolerance =", tolerance)
print("rank =", rank)
print("nullity =", null_basis.shape[1])
print("\nnull-space basis =\n", null_basis)
print("\nA times the null-space basis =\n", A_rank_deficient @ null_basis)

assert rank + null_basis.shape[1] == A_rank_deficient.shape[1]
assert np.allclose(A_rank_deficient @ null_basis, 0.0)

This $2\times3$ matrix has rank one and nullity two, consistent with rank-nullity: $1+2=3$. Its two rows contain the same directional information.

## 4. Geometry of a nearly singular operator

Consider

$$
H_\epsilon=\begin{bmatrix}1&1\\1&1+\epsilon\end{bmatrix}.
$$

The following plots map the unit circle through $H_\epsilon$. The resulting ellipse has semiaxis lengths equal to the singular values.

In [ ]:
angles = np.linspace(0.0, 2.0 * np.pi, 600)
unit_circle = np.vstack((np.cos(angles), np.sin(angles)))
epsilon_values = [0.5, 0.02]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, epsilon in zip(axes, epsilon_values):
    H = np.array([[1.0, 1.0], [1.0, 1.0 + epsilon]])
    mapped = H @ unit_circle
    _, sigma, _ = np.linalg.svd(H)

    ax.plot(unit_circle[0], unit_circle[1], "--", color="0.55", label="unit circle")
    ax.plot(mapped[0], mapped[1], color="tab:blue", lw=2, label=r"$H_\epsilon$ image")
    ax.axhline(0.0, color="0.75", lw=0.8)
    ax.axvline(0.0, color="0.75", lw=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("first component")
    ax.set_ylabel("second component")
    ax.set_title(
        fr"$\epsilon={epsilon:g}$, "
        fr"$\sigma_1={sigma[0]:.3f}$, $\sigma_2={sigma[1]:.3f}$"
    )
    ax.legend()
    ax.grid(True, alpha=0.25)

fig.suptitle("A nearly singular map flattens the unit circle")
fig.tight_layout()
plt.show()

As $\epsilon$ decreases, the ellipse becomes thin. Inputs along one right-singular-vector direction have little effect on the output, so recovering that component from noisy measurements is difficult.

## 5. Conditioning and noise amplification

We solve $H_\epsilon x=b$ after adding a small fixed measurement error to $b$. Every tested matrix is invertible, but the recovered state becomes unreliable as the condition number grows.

In [ ]:
epsilon_sweep = np.logspace(-5, 0, 80)
x_true = np.array([1.0, -1.0])
measurement_noise = np.array([1.0e-4, -1.0e-4])

condition_numbers = []
relative_errors = []

for epsilon in epsilon_sweep:
    H = np.array([[1.0, 1.0], [1.0, 1.0 + epsilon]])
    b_exact = H @ x_true
    x_estimated = np.linalg.solve(H, b_exact + measurement_noise)
    condition_numbers.append(np.linalg.cond(H))
    relative_errors.append(
        np.linalg.norm(x_estimated - x_true) / np.linalg.norm(x_true)
    )

condition_numbers = np.asarray(condition_numbers)
relative_errors = np.asarray(relative_errors)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(epsilon_sweep, condition_numbers, color="tab:blue", lw=2)
axes[0].set_xlabel(r"$\epsilon$")
axes[0].set_ylabel(r"condition number $\kappa_2(H_\epsilon)$")
axes[0].set_title("Conditioning")

axes[1].loglog(epsilon_sweep, relative_errors, color="tab:blue", lw=2)
axes[1].set_xlabel(r"$\epsilon$")
axes[1].set_ylabel("relative state-estimation error")
axes[1].set_title("Amplification of fixed measurement noise")

for ax in axes:
    ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

An inverse does not create information. When $H_\epsilon$ almost removes one input direction, inversion strongly amplifies measurement error in that direction.

## 6. Constructing Jordan form from a supplied matrix

For a square matrix $A$, choose the columns of $P$ from eigenvectors and generalized eigenvectors. Consistent with the course note, the transformed matrix is

$$
\boxed{J=P^{-1}AP}.
$$

The matrix $J$ is block diagonal, with the vectors in $P$ arranged into Jordan chains. Equivalently, $A=PJP^{-1}$.

For an **exact** matrix, SymPy can construct this decomposition symbolically. The function below accepts any square matrix for which the symbolic eigenvalue problem is tractable. Use integers, rational numbers, or symbolic expressions rather than decimal approximations. The resulting $P$ is not unique: eigenvectors and generalized eigenvectors may be scaled or selected differently while producing an equivalent Jordan form.

SymPy is already available in Google Colab. In another Jupyter environment, install it with `%pip install sympy` if necessary.

In [ ]:
import sympy as sp


def exact_jordan_form(A_entries):
    '''Return exact A, P, J satisfying J = P^{-1}*A*P.'''
    A_exact = sp.Matrix(A_entries)
    if not A_exact.is_square:
        raise ValueError("A must be square")

    P_exact, J_exact = A_exact.jordan_form()
    residual = sp.simplify(P_exact.inv() * A_exact * P_exact - J_exact)
    if residual != sp.zeros(*A_exact.shape):
        raise ArithmeticError("Jordan decomposition verification failed")

    return A_exact, P_exact, J_exact

Use the matrix

$$
A=\begin{bmatrix}3&1\\-1&1\end{bmatrix}.
$$

Its characteristic polynomial is $(\lambda-2)^2$, and it has only one independent eigenvector. The call to `jordan_form` constructs both the change-of-basis matrix and the Jordan matrix.

In [ ]:
A_exact, P_exact, J_exact = exact_jordan_form([
    [3, 1],
    [-1, 1],
])

print("A =")
display(A_exact)
print("P =")
display(P_exact)
print("J =")
display(J_exact)
print("P^{-1} A P =")
display(sp.simplify(P_exact.inv() * A_exact * P_exact))
print("Verification: J = P^{-1} A P is", J_exact == P_exact.inv() * A_exact * P_exact)

For this $2\times2$ example, the first column $v_1$ of $P$ is an eigenvector and the second column $v_2$ is a generalized eigenvector. We can verify the Jordan-chain equations directly.

In [ ]:
lambda_exact = J_exact[0, 0]
v1 = P_exact[:, 0]
v2 = P_exact[:, 1]
shifted = A_exact - lambda_exact * sp.eye(A_exact.rows)

print("eigenvalue =", lambda_exact)
print("v1 =")
display(v1)
print("v2 =")
display(v2)
print("(A - lambda I)v1 =")
display(shifted * v1)
print("(A - lambda I)v2 =")
display(shifted * v2)

assert shifted * v1 == sp.zeros(A_exact.rows, 1)
assert shifted * v2 == v1

To apply the procedure to another matrix, replace the entries passed to `exact_jordan_form`. If the matrix is diagonalizable, $J$ will be diagonal. If it has several defective eigenvalues, $J$ may contain several Jordan blocks.

Do not interpret this symbolic routine as a reliable numerical test for Jordan structure. A tiny floating-point perturbation can split a repeated eigenvalue and change the block structure completely. For numerical state-space calculations, use Schur decompositions and direct routines such as `scipy.linalg.expm`.

## 7. Constructing a higher-order Jordan block

An $n\times n$ Jordan block associated with eigenvalue $\lambda$ is

$$
J_n(\lambda)=\lambda I+N,
$$

where $N$ has ones on its first superdiagonal and zeros elsewhere. The standard basis vectors form a Jordan chain:

$$
Ne_1=0,\qquad Ne_{k+1}=e_k.
$$

The matrix $N$ is nilpotent: $N^n=0$, while $N^{n-1}\ne0$.

In [ ]:
def jordan_block(eigenvalue, size):
    '''Construct one size-by-size Jordan block with a superdiagonal of ones.'''
    if size < 1:
        raise ValueError("size must be a positive integer")
    return eigenvalue * np.eye(size) + np.diag(np.ones(size - 1), k=1)


lambda_j = -1.0
n = 8
J = jordan_block(lambda_j, n)
N = J - lambda_j * np.eye(n)

print(f"J_{n}({lambda_j:g}) =\n", J)
print("\n||N^(n-1)|| =", np.linalg.norm(np.linalg.matrix_power(N, n - 1)))
print("||N^n|| =", np.linalg.norm(np.linalg.matrix_power(N, n)))

# Verify every relation N e_{k+1} = e_k in the Jordan chain.
E = np.eye(n)
chain_errors = [np.linalg.norm(N @ E[:, k] - E[:, k - 1]) for k in range(1, n)]
print("maximum Jordan-chain error =", max(chain_errors))

assert np.linalg.norm(np.linalg.matrix_power(N, n - 1)) > 0.0
assert np.allclose(np.linalg.matrix_power(N, n), 0.0)
assert max(chain_errors) == 0.0

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
image = ax.imshow(J, cmap="coolwarm", vmin=-1.0, vmax=1.0)
ax.set_title(fr"Structure of the ${n}\times {n}$ Jordan block $J_{n}(-1)$")
ax.set_xlabel("column")
ax.set_ylabel("row")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
fig.colorbar(image, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

This construction is reliable because the desired structure is specified exactly. It is different from asking a floating-point algorithm to decide whether an arbitrary matrix has exact repeated eigenvalues and exact Jordan chains.

## 8. Exponential of a large Jordan block

Since $J=\lambda I+N$, the matrices $\lambda I$ and $N$ commute. Because $N^n=0$, the exponential series terminates:

$$
e^{Jt}=e^{\lambda t}
\sum_{k=0}^{n-1}\frac{t^k}{k!}N^k.
$$

We compare this finite Jordan formula with `scipy.linalg.expm`.

In [ ]:
def jordan_exponential(eigenvalue, size, t):
    N = np.diag(np.ones(size - 1), k=1)
    finite_series = np.zeros((size, size))
    for k in range(size):
        finite_series += (t**k / factorial(k)) * np.linalg.matrix_power(N, k)
    return np.exp(eigenvalue * t) * finite_series


t_check = 1.25
exp_from_jordan_series = jordan_exponential(lambda_j, n, t_check)
exp_from_scipy = expm(J * t_check)
comparison_error = np.linalg.norm(exp_from_jordan_series - exp_from_scipy)

print("finite-series result =\n", exp_from_jordan_series)
print("\ncomparison error =", comparison_error)

assert np.allclose(exp_from_jordan_series, exp_from_scipy)

To expose every polynomial factor, choose the last generalized eigenvector $e_n$ as the initial condition. The first state then contains $t^{n-1}e^{\lambda t}/(n-1)!$, the second contains $t^{n-2}e^{\lambda t}/(n-2)!$, and so on.

In [ ]:
time = np.linspace(0.0, 12.0, 500)
x0 = np.eye(n)[:, -1]
x_history = np.column_stack([jordan_exponential(lambda_j, n, ti) @ x0 for ti in time])

fig, ax = plt.subplots(figsize=(9, 5))
for index in range(n):
    power = n - index - 1
    ax.plot(time, x_history[index], lw=1.7, label=fr"$x_{index + 1}$: $t^{power}e^{{-t}}/{power}!$")

ax.set_xlabel("time")
ax.set_ylabel("state component")
ax.set_title("Polynomial-exponential terms generated by an 8-by-8 Jordan block")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### Several Jordan blocks

A Jordan matrix may contain blocks of different sizes and eigenvalues. `scipy.linalg.block_diag` can assemble such a matrix when the block structure is already known.

In [ ]:
J_complete = block_diag(
    jordan_block(-1.0, 4),
    jordan_block(-2.0, 2),
    jordan_block(0.5, 3),
)

print("Jordan matrix assembled from blocks =\n", J_complete)
print("shape =", J_complete.shape)

## 9. Student investigations

1. Set the initial displacement of the two-mass system equal to one eigenvector. Which modal coordinate remains zero?
2. Replace $H_\epsilon$ with $\begin{bmatrix}1&2\\1&2+\epsilon\end{bmatrix}$. Does the same conditioning trend remain?
3. Change the direction of the measurement noise. Which noise directions are amplified most strongly?
4. Replace the entries in `exact_jordan_form` with another exact square matrix and verify $J=P^{-1}AP$.
5. Change the Jordan-block size from 8 to 3, 5, and 12. Verify the nilpotency index each time.
6. Change the Jordan eigenvalue from $-1$ to $0$ and $0.2$. How does the response change?
7. Start the Jordan system at $e_1$ instead of $e_n$. Which polynomial factors disappear?
8. Perturb one diagonal entry of the Jordan block by $10^{-8}$ and inspect the eigenvalues. Why does this illustrate the numerical fragility of Jordan form?

## 10. Takeaways

- A coordinate transformation can expose structure without changing the underlying operator.
- Eigenvectors decouple a diagonalizable operator into modal directions.
- The SVD reveals rank and the directions of strongest and weakest amplification.
- A small singular value makes inverse calculations sensitive to noise.
- For exact square matrices, symbolic software can construct $P$ and $J$ and verify $J=P^{-1}AP$.
- A Jordan chain is represented by the nilpotent superdiagonal of a Jordan block.
- Larger Jordan blocks generate higher-degree polynomial factors multiplying $e^{\lambda t}$.
- Constructing a known Jordan block is straightforward; inferring Jordan form from floating-point data is not numerically reliable.
- For routine numerical dynamics, use stable functions such as `scipy.linalg.expm` rather than an explicitly computed Jordan decomposition.